In [ ]:
# Quantum simulation libraries
from qutip import (
    basis, 
    mesolve, 
    qeye, 
    sigmax, 
    sigmay, 
    sigmaz, 
    tensor,
    )
import qutip

# Machine learning libraries
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

# Plotting libraries
import matplotlib.pyplot as plt
from rich.progress import track
import seaborn as sns

# Linalg libraries
import numpy as np
from scipy.stats import pearsonr

# Helper libraries
from dataclasses import dataclass


In [ ]:
# set the device
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

# Load target data

In [ ]:
class BFieldGenerator:
    def __init__(
        self,  amplitude: float, frequency: float, resolution: int = 0.01, length: int = 1000
        ):
        """
        Create the BField generator.

        Parameters
        ----------
        amplitude : float
            Amplitude of the magnetic field.
        frequency : float
            Frequency of the cosine wave.
        resoltion : float
            Distance between measured points.
        length : int
            Amount of time to run it for.
        """
        self.amplitude = amplitude
        self.frequency = frequency

        self.counter = 0

        self.measured_field = []
        self.times = []

    def __call__(self, t: float):
        """
        Get the next value of the magnetic field.

        Parameters
        ----------
        t : float
            Current time, but this is ignored.
        """
        b_field = self.amplitude * np.cos(self.frequency * t)
        self.measured_field.append(b_field)
        self.times.append(t)
        self.counter += 1

        return b_field

# Run Simulation

In [ ]:
@dataclass
class SimulationParameters:
    """
    Helper class for the simulation parameters.
    """
    length: int
    coupling: list

@dataclass
class SimulationState:
    """ 
    Helper class for the simulation state.
    """
    number_of_spins: int
    quantum_state: list
    spin_list: list
    coupling_list: list

In [ ]:
def get_simulation_state(parameters: SimulationParameters):
    """
    Returns the initial state of the simulation.
    """
    # Get the initial wavefunction
    number_of_spins = parameters.length

    initial_state = [basis(2, 1)] + [basis(2, 0)] * (number_of_spins - 1)

    # Setup operators for individual qubits
    sx_list, sy_list, sz_list = [], [], []
    for i in range(number_of_spins):
        op_list = [qeye(2)] * number_of_spins
        op_list[i] = sigmax()
        sx_list.append(tensor(op_list))
        op_list[i] = sigmay()
        sy_list.append(tensor(op_list))
        op_list[i] = sigmaz()
        sz_list.append(tensor(op_list))

    # Setup the operators for the coupling
    Jx = parameters.coupling * np.ones(number_of_spins)
    Jy = parameters.coupling * np.ones(number_of_spins)
    Jz = parameters.coupling * np.ones(number_of_spins)    

    return SimulationState(
        number_of_spins=number_of_spins,
        quantum_state=tensor(initial_state),
        spin_list=[sx_list, sy_list, sz_list],
        coupling_list=[Jx, Jy, Jz],
    )

In [ ]:
def compute_hamiltonian(t, args):
    """
    Compute the Hamiltonian at time t.
    
    Parameters
    ----------
    t : float
        Current time.
    args : dict
        System parameters in the H computation.
    """
    sx_list = args["sx_list"]
    sy_list = args["sy_list"]
    sz_list = args["sz_list"]
    Jx = args["Jx"]
    Jy = args["Jy"]
    Jz = args["Jz"]
    driving_field = args["driving"]

    N = args["N"]

    # Magnetic field terms to top row
    H = 0
    for i in [0, 1, 3]:        
        H -= driving_field(t) * sz_list[i]

    
    # Interaction terms
    for n in range(N - 1):
        H += -0.5 * Jx[n] * sx_list[n] * sx_list[n + 1]
        H += -0.5 * Jy[n] * sy_list[n] * sy_list[n + 1]
        H += -0.5 * Jz[n] * sz_list[n] * sz_list[n + 1]

    return H

In [ ]:
# for strength in np.linspace(1, 10, 10, dtype=int):
simulation_parameters = SimulationParameters(
    length=5,
    coupling=10. * np.pi,
)
simulation_state = get_simulation_state(simulation_parameters)

field_generator = BFieldGenerator(1.0, 0.1 * np.pi, length=20)
args = {
    "sx_list": simulation_state.spin_list[0],
    "sy_list": simulation_state.spin_list[1],
    "sz_list": simulation_state.spin_list[2],
    "Jx": simulation_state.coupling_list[0],
    "Jy": simulation_state.coupling_list[1],
    "Jz": simulation_state.coupling_list[2],
    "N": simulation_state.number_of_spins,
    "driving": field_generator
}

In [ ]:
times = np.linspace(0, 500, 10000)
results = mesolve(compute_hamiltonian, simulation_state.quantum_state, times, [], [], args)

# qutip.qsave(states, f"{strength}")

In [ ]:
fit_field = BFieldGenerator(1.0, 0.1 * np.pi, length=20)

In [ ]:
b_field = fit_field(results.times)

In [ ]:
plt.plot(results.times, b_field)

In [ ]:
def generate_gue_matrix(size, dims):
    # Generate a random complex matrix with Gaussian-distributed entries
    real_part = np.random.normal(scale=1/np.sqrt(2), size=(size, size))
    imag_part = np.random.normal(scale=1/np.sqrt(2), size=(size, size))
    random_matrix = real_part + 1j * imag_part

    # Symmetrize the matrix to make it Hermitian
    hermitian_matrix = (random_matrix + random_matrix.conj().T) / 2

    return qutip.Qobj(hermitian_matrix, dims=dims)

In [ ]:
state_dimension = 300

measurements = []
# tensor_structure = [[2, 2, 2, 2, 2], [2, 2, 2, 2, 2]]
# size = np.prod(tensor_structure[0])

for _ in range(state_dimension):
    # gue_matrix = generate_gue_matrix(size, tensor_structure)
    non_gue_matrix = qutip.rand_herm(
        32, 
        0.9, 
        dims=[[2, 2, 2, 2, 2], [2, 2, 2, 2, 2]]
    )

    # measurements.append(gue_matrix)
    measurements.append(non_gue_matrix)

In [ ]:
# for strength in np.linspace(1, 10, 10, dtype=int):

# Compute observables
observations = np.zeros((10000, state_dimension))
# states = qutip.qload(f"{strength}")
# Calculate the size of the matrix for the given tensor structure
states = results.states

for t, state in enumerate(states):
    for o, operator in enumerate(measurements):
        # measure_state = state.ptrace([1, 2, 4])
        measure_state = state
        observations[t][o] = qutip.expect(measure_state * measure_state.dag(), operator)

    # np.save(f"{strength}_observations.npy", observations)

# Fit Model

In [ ]:
class NeuralNetwork(nn.Module):
    
    def __init__(self, state_dimension: int, output_dimension: int):
        """
        Build the network.
        
        Parameters
        ----------
        state_dimension : int
                Dimension of the state representation.
                This is the input to the layer.
        output_dimension : int
                Dimension of the output being predicted.
        """
        super().__init__()


        # self.readout_layer = nn.Sequential(
        #     nn.Linear(state_dimension, output_dimension),
        # )
        self.linear_stack = nn.Sequential(
            nn.Linear(state_dimension, 128),
            nn.ReLU(),
            nn.Linear(128, output_dimension)
        )
    
    def forward(self, x):
        """
        Forward pass through the network.
        
        As we are doing reservoir computing, this is 
        simply a linear layer.
        """
        # return self.readout_layer(x)
        return self.linear_stack(x)

In [ ]:
class ReservoirDataset(Dataset):
    """
    Custom dataset for the training.
    """
    def __init__(
        self, 
        state_data: np.ndarray, 
        prediction_length: int,
        function_data: np.ndarray,
    ):
        """
        Constructor for the dataset.

        Parameters
        ----------
        state_data : np.ndarray
                State description data.
        function_data : np.ndarray
                Function data being fit.
                This will be the target data.
        prediction_length : int
                How far into the future you will predict.
        """
        self.state_data = torch.Tensor(state_data[:-prediction_length]).to(device)

        self.function_data = torch.Tensor(
            function_data[prediction_length:].reshape(-1, 1)
        ).to(device)
        
        self.norm_factor = 1 # max(abs(function_data.flatten()))

    def split_data(self, train_size: float):
        """
        Split the data into training and validation sets.
        
        Parameters
        ----------
        train_size : float
                Size of the training set.
        """
        train_size = int(train_size * len(self))
        val_size = len(self) - train_size
        return torch.utils.data.random_split(self, [train_size, val_size])
    
    def __len__(self):
        """
        Length of the dataset.
        """
        return int(
            len(self.function_data)
        )
    
    def __getitem__(self, idx: int):
        """
        Collect an item from the dataset.
        
        Parameters
        ----------
        idx : int
                Index of the state to take.
        """
        state = self.state_data[idx]
        target = self.function_data[idx] / self.norm_factor
        
        return state, target

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss = loss.item()
            
    return loss

In [ ]:
def test(dataloader, model, loss_fn) -> np.ndarray:
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss = 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
    test_loss /= num_batches
    
    return test_loss

In [ ]:
def train_model(dataset, test_ds, model = None):
    """
    Train a model on the current data.
    """    
    if model is None:
        model = NeuralNetwork(
            state_dimension=300,
            output_dimension=1
        ).to(device)

        model = model.type(torch.float32)

    # Use MSE loss
    loss_fn = nn.MSELoss()

    # Use ADAM optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

    # Create the loader
    loader = DataLoader(dataset, batch_size=250, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=250, shuffle=False)
    
    # Train the network
    epochs = 2000
    train_losses = []
    test_losses = []

    for t in track(range(epochs)):
        loss = train(loader, model, loss_fn, optimizer)
        train_losses.append(loss)
        loss = test(test_loader, model, loss_fn)
        test_losses.append(loss)

    train_losses = [item.cpu().detach().numpy() for item in train_losses]
    
    return train_losses, test_losses, model

## Train the model

In [ ]:
prediction_length = 1
split_offset=7000

train_ds = ReservoirDataset(
    state_data=observations[:split_offset, :],
    function_data=b_field[:split_offset],
    prediction_length=prediction_length
)

test_ds = ReservoirDataset(
    state_data=observations[split_offset:, :],
    function_data=b_field[split_offset:],
    prediction_length=prediction_length
)

train_losses, test_losses, model = train_model(train_ds, test_ds, model=None)



In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5))

ax[0].plot(test_losses)
ax[1].plot(train_losses)

ax[1].set_yscale("log")
ax[0].set_yscale("log")

plt.show()

In [ ]:
test_predictions = []
test_targets = []
for item in test_ds:
    state, target = item
    test_predictions.append(model(state).cpu().detach().numpy())
    test_targets.append(target.cpu().detach().numpy())

test_predictions = np.array(test_predictions).reshape(-1)
test_targets = np.array(test_targets).reshape(-1)

train_predictions = []
train_targets = []
for item in train_ds:
    state, target = item
    train_predictions.append(model(state).cpu().detach().numpy())
    train_targets.append(target.cpu().detach().numpy())

train_predictions = np.array(train_predictions).reshape(-1)
train_targets = np.array(train_targets).reshape(-1)

In [ ]:
plt.plot(results.times, b_field, label="baseline")

plt.plot(
    results.times[prediction_length:split_offset], 
    train_predictions, 
    "o",
    label="train", 
    mfc="none", 
    markersize=2
)

plt.plot(
    results.times[prediction_length + split_offset:], 
    test_predictions, 
    "o",
    label="test", 
    mfc="none", 
    markersize=2
)
plt.tight_layout()
plt.legend()
plt.xlabel("Time")
plt.ylabel("B Field")

plt.savefig("strongly-correlated-predictions.pdf")
plt.show()


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5))

ax[0].plot(test_targets, test_predictions, "k.", label=pearsonr(test_targets, test_predictions)[0])
ax[0].plot(test_targets, test_targets, "r--")
ax[0].legend()

ax[1].plot(train_targets, train_predictions, "k.", label=pearsonr(train_targets, train_predictions)[0])
ax[1].plot(train_targets, train_targets, "r--")
ax[1].legend()

# fig.legend()
plt.show()

In [ ]:
test_error_deltas = np.abs(test_targets - test_predictions)
train_error_deltas = np.abs(train_targets - train_predictions)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))

ax[0].plot(test_targets, test_error_deltas)
ax[1].plot(train_targets, train_error_deltas)

# ax[0].set_yscale("log")
# ax[1].set_yscale("log")

plt.show()